1. Introducción

Leicester City conquistó la Premier League 2015/16 gracias a una combinación de solidez defensiva y eficacia ofensiva. Aunque no lideró ninguna de las dos estadísticas principales, se ubicó entre los mejores equipos tanto en goles anotados como en goles recibidos. Esta combinación de equilibrio y regularidad le permitió sumar puntos de manera constante durante toda la temporada.

Además, el dato más diferencial fue su capacidad para evitar derrotas: perdió únicamente 3 de los 38 partidos disputados, mientras que sus principales perseguidores sufrieron entre 6 y 10 derrotas. Esto sugiere que la clave de su campeonato no fue dominar una métrica específica, sino mantener un rendimiento alto y sostenido a lo largo de toda la temporada.

Cargamos el dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/data/season-1516_csv.csv")

df.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
0,E0,08/08/15,Bournemouth,Aston Villa,0,1,A,0,0,D,...,1.79,26,-0.5,1.98,1.93,1.99,1.92,1.82,3.88,4.70
1,E0,08/08/15,Chelsea,Swansea,2,2,D,2,1,H,...,1.99,27,-1.5,2.24,2.16,1.80,1.73,1.37,5.04,10.88
2,E0,08/08/15,Everton,Watford,2,2,D,0,1,A,...,1.96,26,-1.0,2.28,2.18,1.76,1.71,1.75,3.76,5.44
3,E0,08/08/15,Leicester,Sunderland,4,2,H,3,0,H,...,1.67,26,-0.5,2.00,1.95,1.96,1.90,1.79,3.74,5.10
4,E0,08/08/15,Man United,Tottenham,1,0,H,1,0,H,...,2.01,26,-1.0,2.20,2.09,1.82,1.78,1.64,4.07,6.04


Limpiamos y renombramos

In [2]:
columnas = {
    "Div": "division",
    "Date": "match_date",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR": "full_time_result",
    "HTHG": "home_half_time_goals",
    "HTAG": "away_half_time_goals",
    "HTR": "half_time_result",
    "Referee": "referee",
    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_shots_on_target",
    "AST": "away_shots_on_target",
    "HF": "home_fouls",
    "AF": "away_fouls",
    "HC": "home_corners",
    "AC": "away_corners",
    "HY": "home_yellow_cards",
    "AY": "away_yellow_cards",
    "HR": "home_red_cards",
    "AR": "away_red_cards"
}

df = df.rename(columns=columnas)

df_analysis = df[
    [
        "match_date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "full_time_result",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_corners",
        "away_corners",
        "home_yellow_cards",
        "away_yellow_cards"
    ]
]



Creamos el dataset que nos importa

In [3]:
leicester = df_analysis[
    (df_analysis["home_team"] == "Leicester") |
    (df_analysis["away_team"] == "Leicester")
].copy()

leicester.head()

,match_date,home_team,away_team,home_goals,away_goals,full_time_result,home_shots,away_shots,home_shots_on_target,away_shots_on_target,home_corners,away_corners,home_yellow_cards,away_yellow_cards
3,08/08/15,Leicester,Sunderland,4,2,H,19,10,8,5,6,3,2,4
16,15/08/15,West Ham,Leicester,1,2,A,10,11,3,6,8,4,1,3
21,22/08/15,Leicester,Tottenham,1,1,D,13,19,2,6,2,7,1,4
31,29/08/15,Bournemouth,Leicester,1,1,D,5,4,2,2,4,4,2,4
47,13/09/15,Leicester,Aston Villa,3,2,H,21,11,6,4,4,5,2,1


2. Rendimiento general del Leicester

Calculamos victorias, empates y derrotas

In [4]:
def leicester_result(row):
    if row["home_team"] == "Leicester":
        if row["full_time_result"] == "H":
            return "Win"
        elif row["full_time_result"] == "D":
            return "Draw"
        else:
            return "Loss"
    else:
        if row["full_time_result"] == "A":
            return "Win"
        elif row["full_time_result"] == "D":
            return "Draw"
        else:
            return "Loss"


def calcular_puntos_partido(row):
    if row["result"] == "Win":
        return 3
    elif row["result"] == "Draw":
        return 1
    else:
        return 0

leicester["result"] = leicester.apply(leicester_result, axis=1)
leicester["result"].value_counts()

leicester["points"] = leicester.apply(calcular_puntos_partido, axis=1)
puntos_totales = leicester["points"].sum()

print("Puntos Totales:", puntos_totales , "\nPartidos ganados:", leicester["result"].value_counts().get("Win", 0), "\nPartidos empatados:", leicester["result"].value_counts().get("Draw", 0), "\nPartidos perdidos:", leicester["result"].value_counts().get("Loss", 0)    )

Puntos Totales: 81 
Partidos ganados: 23 
Partidos empatados: 12 
Partidos perdidos: 3


Goles a favor y en contra

In [5]:
def calcular_goles_favor(row):
    if row["home_team"] == "Leicester":
        return row["home_goals"]
    else:
        return row["away_goals"]

def calcular_goles_contra(row):
    if row["home_team"] == "Leicester":
        return row["away_goals"]
    else:
        return row["home_goals"]

leicester["goals_for"] = leicester.apply(calcular_goles_favor, axis=1)
leicester["goals_against"] = leicester.apply(calcular_goles_contra, axis=1)

print("Goals For:", leicester["goals_for"].sum())
print("Goals Against:", leicester["goals_against"].sum())

Goals For: 68
Goals Against: 36


Resumen:

In [6]:
wins = (leicester["result"] == "Win").sum()
draws = (leicester["result"] == "Draw").sum()
losses = (leicester["result"] == "Loss").sum()
puntos_totales = leicester["points"].sum()

print(f"Wins: {wins}")
print(f"Draws: {draws}")
print(f"Losses: {losses}")
print("Puntos Totales:", puntos_totales)
print("Goals For:", leicester["goals_for"].sum())
print("Goals Against:", leicester["goals_against"].sum())

Wins: 23
Draws: 12
Losses: 3
Puntos Totales: 81
Goals For: 68
Goals Against: 36


Conclusión:

El Leicester City finalizó la temporada 2015/16 con 81 puntos, producto de 23 victorias, 12 empates y solo 3 derrotas en 38 partidos. Este rendimiento demuestra una gran consistencia a lo largo de toda la temporada, ya que el equipo evitó las derrotas en 35 de los 38 encuentros disputados.

En el aspecto ofensivo, marcó 68 goles, mientras que defensivamente recibió 36, logrando una diferencia de gol de +32. Si bien estos números reflejan un equipo sólido tanto en ataque como en defensa, lo más destacable es su capacidad para sumar puntos de manera constante, incluso en partidos donde no conseguía la victoria.

Los datos sugieren que el éxito del Leicester no se debió únicamente a una gran capacidad goleadora, sino principalmente a su regularidad y a su habilidad para mantenerse competitivo durante toda la temporada, características fundamentales para conquistar la Premier League.

3. Comparación con los principales competidores

Creamos una tabla de posiciones completa de la Premier League 2015/16 para comparar el rendimiento del Leicester con otros equipos.

In [7]:
teams = pd.concat([
    df_analysis["home_team"],
    df_analysis["away_team"]
]).unique()


table = pd.DataFrame({
    "position": 0,
    "team": teams,
    "points": 0,
    "wins": 0,
    "draws": 0,
    "losses": 0,
    "goals_for": 0,
    "goals_against": 0
})


for team in teams:

    matches = df_analysis[
        (df_analysis["home_team"] == team) |
        (df_analysis["away_team"] == team)
    ]

    wins = draws = losses = 0
    goals_for = goals_against = 0

    for _, row in matches.iterrows():

        if row["home_team"] == team:

            goals_for += row["home_goals"]
            goals_against += row["away_goals"]

            if row["full_time_result"] == "H":
                wins += 1
            elif row["full_time_result"] == "D":
                draws += 1
            else:
                losses += 1

        else:

            goals_for += row["away_goals"]
            goals_against += row["home_goals"]

            if row["full_time_result"] == "A":
                wins += 1
            elif row["full_time_result"] == "D":
                draws += 1
            else:
                losses += 1

    points = wins * 3 + draws

    table.loc[table["team"] == team, "points"] = points
    table.loc[table["team"] == team, "wins"] = wins
    table.loc[table["team"] == team, "draws"] = draws
    table.loc[table["team"] == team, "losses"] = losses
    table.loc[table["team"] == team, "goals_for"] = goals_for
    table.loc[table["team"] == team, "goals_against"] = goals_against

    table["goal_difference"] = (table["goals_for"] - table["goals_against"]
    )


Primer hallazgo importante:

Leicester ganó la liga por 10 puntos de diferencia sobre Arsenal. En la Premier League, 10 puntos es una ventaja considerable.

In [8]:
table = table.sort_values(by=["points", "goal_difference"],ascending=False).reset_index(drop=True)
table["position"] = table.index + 1
    
print(table.head(10).to_string(index=False))

 position        team  points  wins  draws  losses  goals_for  goals_against  goal_difference
        1   Leicester      81    23     12       3         68             36               32
        2     Arsenal      71    20     11       7         65             36               29
        3   Tottenham      70    19     13       6         69             35               34
        4    Man City      66    19      9      10         71             41               30
        5  Man United      66    19      9      10         49             35               14
        6 Southampton      63    18      9      11         59             41               18
        7    West Ham      62    16     14       8         65             51               14
        8   Liverpool      60    16     12      10         63             50               13
        9       Stoke      51    14      9      15         41             55              -14
       10     Chelsea      50    12     14      12         5

Segundo hallazgo:

Leicester NO fue el equipo más goleador. Si bien quedó a 3 goles de igualar al Manchester City, no se puede decir que fue campeón gracias a su cantidad de goles.


In [9]:
goals_for= table.sort_values(by="goals_for", ascending=False)
goals_for= goals_for[["position", "team", "goals_for"]]
print(goals_for.head(10).to_string(index=False))

 position        team  goals_for
        4    Man City         71
        3   Tottenham         69
        1   Leicester         68
        2     Arsenal         65
        7    West Ham         65
        8   Liverpool         63
       10     Chelsea         59
       11     Everton         59
        6 Southampton         59
        5  Man United         49


Tercer hallazgo:

Leicester NO fue la mejor defensa pero tuvo muy pocos goles encajados. Por lo tanto, se puede decir que su solidez defensiva fue clave para ganar el título.

In [10]:
goals_against= table.sort_values(by="goals_against", ascending=False)
goals_against= goals_against[["position", "team", "goals_against"]]
print(goals_against.to_string(index=False))

 position           team  goals_against
       20    Aston Villa             76
       19        Norwich             67
       16    Bournemouth             67
       18      Newcastle             65
       17     Sunderland             62
        9          Stoke             55
       11        Everton             55
       10        Chelsea             53
       12        Swansea             52
        7       West Ham             51
       15 Crystal Palace             51
        8      Liverpool             50
       13        Watford             50
       14      West Brom             48
        6    Southampton             41
        4       Man City             41
        2        Arsenal             36
        1      Leicester             36
        5     Man United             35
        3      Tottenham             35


Conclusión:

Leicester City conquistó la Premier League 2015/16 gracias a una combinación de solidez defensiva y eficacia ofensiva. Aunque no lideró ninguna de las dos estadísticas principales, se ubicó entre los mejores equipos tanto en goles anotados como en goles recibidos. Esta combinación de equilibrio y regularidad le permitió sumar puntos de manera constante durante toda la temporada.

Además, el dato más diferencial fue su capacidad para evitar derrotas: perdió únicamente 3 de los 38 partidos disputados, mientras que sus principales perseguidores sufrieron entre 6 y 10 derrotas. Esto sugiere que la clave de su campeonato no fue dominar una métrica específica, sino mantener un rendimiento alto y sostenido a lo largo de toda la temporada.

4. ¿Fortaleza de local o de visitante?

Separamos los partidos de local y los de visitante

In [11]:
home_matches = leicester[
    leicester["home_team"] == "Leicester"
].copy()

away_matches = leicester[
    leicester["away_team"] == "Leicester"
].copy()

In [12]:
print(len(home_matches))
print(len(away_matches))

19
19


Calculamos los puntos, goles a favor, goles en contra y victorias de local.

In [15]:
home_points = 0

for _, row in home_matches.iterrows():

    if row["full_time_result"] == "H":
        home_points += 3

    elif row["full_time_result"] == "D":
        home_points += 1

print("Home Points:", home_points)

home_goals_for = home_matches["home_goals"].sum()
home_goals_against = home_matches["away_goals"].sum()

print("Home Goals For:", home_goals_for)
print("Home Goals Against:", home_goals_against)

home_wins = (
    home_matches["full_time_result"] == "H"
).sum()

print("Home Wins:", home_wins)

Home Points: 42
Home Goals For: 35
Home Goals Against: 18
Home Wins: 12


Calculamos los puntos, goles a favor, goles en contra y victorias de visitante.

In [16]:
away_points = 0

for _, row in away_matches.iterrows():

    if row["full_time_result"] == "A":
        away_points += 3

    elif row["full_time_result"] == "D":
        away_points += 1

print("Away Points:", away_points)

away_goals_for = away_matches["away_goals"].sum()
away_goals_against = away_matches["home_goals"].sum()

print("Away Goals For:", away_goals_for)
print("Away Goals Against:", away_goals_against)

away_wins = (
    away_matches["full_time_result"] == "A"
).sum()

print("Away Wins:", away_wins)

Away Points: 39
Away Goals For: 33
Away Goals Against: 18
Away Wins: 11


Conclusión:

Uno de los aspectos más destacados de la temporada 2015/16 fue la consistencia del Leicester City tanto en casa como fuera de ella. El equipo obtuvo 42 puntos como local y 39 como visitante, mostrando un rendimiento prácticamente idéntico en ambos escenarios.

Además, mantuvo el mismo nivel defensivo al recibir únicamente 18 goles tanto en condición de local como de visitante. En ataque también presentó cifras muy similares, con 35 goles anotados en casa y 33 fuera de ella.

Estos resultados sugieren que Leicester no fue un equipo dependiente de su estadio, sino un conjunto capaz de competir y obtener resultados positivos en cualquier contexto, una característica poco habitual en los equipos de la talla del Leicester, pero un aspecto clave para ser campeones de liga.

5. Efectividad de cara al arco

Guardamos los datos de disparos y disparos que van al arco.

In [17]:
leicester["shots"] = leicester.apply(
    lambda row: row["home_shots"]
    if row["home_team"] == "Leicester"
    else row["away_shots"],
    axis=1
)

leicester["shots_on_target"] = leicester.apply(
    lambda row: row["home_shots_on_target"]
    if row["home_team"] == "Leicester"
    else row["away_shots_on_target"],
    axis=1
)

print("Goals:", leicester["goals_for"].sum())
print("Shots:", leicester["shots"].sum())
print("Shots on Target:", leicester["shots_on_target"].sum())

Goals: 68
Shots: 523
Shots on Target: 181


Métricas de eficiencia:

Conversión de tiros, conversión de tiros al arco y precisión de tiro cada 100 tiros, y tiros por partido.

In [22]:
conversion_rate = (
    leicester["goals_for"].sum()
    / leicester["shots"].sum()
)

print("Conversion Rate:", round(conversion_rate * 100, 2), "%")


shot_on_target_conversion = (
    leicester["goals_for"].sum()
    / leicester["shots_on_target"].sum()
)

print("Shot on Target Conversion:", round(shot_on_target_conversion * 100, 2), "%"  )


accuracy = (
    leicester["shots_on_target"].sum()
    / leicester["shots"].sum()
)

print("Accuracy:", round(accuracy * 100, 2), "%")

shots_per_match = leicester["shots"].sum() / len(leicester)
print("Shots per Match:", round(shots_per_match, 2))


Conversion Rate: 13.0 %
Shot on Target Conversion: 37.57 %
Accuracy: 34.61 %
Shots per Match: 13.76


Todavía no podemos afirmar si Leicester fue extremadamente eficiente. ¿13.0 % de tasa de conversión es mucho o poco para la Premier League 2015/16?

Vamos a responderlo comparandolo con sus perseguidores.

In [30]:
def team_stats(team_name):

    team = df_analysis[
        (df_analysis["home_team"] == team_name) |
        (df_analysis["away_team"] == team_name)
    ].copy()

    team["goals_for"] = team.apply(
        lambda row: row["home_goals"]
        if row["home_team"] == team_name
        else row["away_goals"],
        axis=1
    )

    team["shots"] = team.apply(
        lambda row: row["home_shots"]
        if row["home_team"] == team_name
        else row["away_shots"],
        axis=1
    )

    team["shots_on_target"] = team.apply(
        lambda row: row["home_shots_on_target"]
        if row["home_team"] == team_name
        else row["away_shots_on_target"],
        axis=1
    )

    goals = team["goals_for"].sum()
    shots = team["shots"].sum()
    shots_on_target = team["shots_on_target"].sum()

    return {
        "team": team_name,
        "goals": goals,
        "shots": shots,
        "shots_on_target": shots_on_target,
        "conversion_rate": round(goals / shots * 100, 2),
        "accuracy": round(shots_on_target / shots * 100, 2),
        "shot_on_target_conversion": round(goals / shots_on_target * 100, 2)
    }

In [31]:
teams = [
    "Leicester",
    "Arsenal",
    "Tottenham",
    "Man City",
    "Man United"
]

results = []

for team in teams:
    results.append(team_stats(team))

comparison = pd.DataFrame(results)

comparison

,team,goals,shots,shots_on_target,conversion_rate,accuracy,shot_on_target_conversion
0,Leicester,68,523,181,13.00,34.61,37.57
1,Arsenal,65,572,210,11.36,36.71,30.95
2,Tottenham,69,661,250,10.44,37.82,27.60
3,Man City,71,612,210,11.60,34.31,33.81
4,Man United,49,430,143,11.40,33.26,34.27


Conclusión:

El análisis de eficiencia ofensiva revela uno de los factores clave detrás del título del Leicester City. Aunque no fue el equipo que más remató ni el que más ocasiones generó, sí fue el más eficiente de los principales candidatos al campeonato.

Leicester convirtió el 13% de sus tiros en goles, la mejor tasa entre los equipos analizados. Además, el 37.57% de sus tiros al arco terminaron en gol, superando a Arsenal, Manchester City y Tottenham.

La comparación con Tottenham resulta especialmente interesante. Mientras que Tottenham registró 661 tiros y 250 tiros al arco, las cifras más altas entre los equipos analizados, Leicester produjo solo 523 tiros y 181 tiros al arco. Sin embargo, ambos equipos terminaron la temporada con una cantidad de goles muy similar (69 para Tottenham y 68 para Leicester). Esto demuestra que Leicester necesitó muchas menos oportunidades para alcanzar prácticamente la misma producción ofensiva.

Estos resultados sugieren que el éxito del Leicester no estuvo basado en un volumen superior de ataques, sino en una notable capacidad para aprovechar las oportunidades generadas. Su elevada eficacia frente al arco rival le permitió maximizar el rendimiento de cada ocasión creada, convirtiéndose en uno de los factores determinantes de su histórica conquista de la Premier League 2015/16.